# RULER trace answer ranking (LangSmith)

This notebook pulls **the same LangSmith root runs** as `ch09_example_external_evaluation_pipelines_langsmith.ipynb`: same **project**, **tag** (`ext_eval_pipelines`), **filters**, and **payload parsing**.

**Note:** `ch09_example_external_evaluation_pipelines_new_.ipynb` is the **Langfuse** cookbook. RULER here only sees runs in **LangSmith**. Use the LangSmith external-eval notebook (or duplicate logging to LangSmith) so traces land in the same project.


# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_Session_4_ruler_trace_answer_ranking_langsmith.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





In [ ]:
%pip install langsmith openpipe-art nest-asyncio python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.0/332.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.0/955.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.4 MB/s eta 0:00:00


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# US-hosted LangSmith (default). For EU, set LANGCHAIN_ENDPOINT=https://eu.api.smith.langchain.com
LS_API_URL = os.getenv("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")
os.environ["LANGCHAIN_ENDPOINT"] = LS_API_URL

api_key = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")
if not api_key:
    raise ValueError(
        "Set LANGCHAIN_API_KEY (or LANGSMITH_API_KEY) in your environment before running this notebook."
    )
os.environ["LANGCHAIN_API_KEY"] = api_key
os.environ["LANGSMITH_API_KEY"] = api_key

# LangSmith project for this cookbook — set here (same as LangSmith external-eval notebook).
# Use "default" unless you created a dedicated project in the UI first.
LANGSMITH_PROJECT_NAME = "default"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT_NAME

# Enable tracing for @traceable (mirrors LangSmith external-eval notebook)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_TRACING_V2"] = "true"


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not found.")
    OPENAI_API_KEY = input("Enter OPENAI_API_KEY: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY is set.")
print("LangSmith credentials are set.")
print(f"LangSmith project (this notebook): {os.environ['LANGCHAIN_PROJECT']}")


OPENAI_API_KEY is set.
LangSmith credentials are set.
LangSmith project (this notebook): default


In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from collections import defaultdict

from langsmith import Client

BATCH_SIZE = 10
TOTAL_TRACES = 100
EVAL_TAG = "ext_eval_pipelines"
WORKFLOW_FILTERS = {"billing", "shipping", "returns", "account_access"}
AGENT_TYPE_FILTERS = {"triage_agent", "billing_agent", "returns_agent"}

LS_API_URL = os.environ["LANGCHAIN_ENDPOINT"]
PROJECT_NAME = os.environ["LANGCHAIN_PROJECT"]
ls_client = Client(api_url=LS_API_URL)

now = datetime.now(timezone.utc)
five_am_today = datetime(now.year, now.month, now.day, 5, 0, tzinfo=timezone.utc)
five_am_yesterday = five_am_today - timedelta(days=1)
seven_days_ago = now - timedelta(days=7)


def parse_trace_payload(run):
    out = run.outputs
    if out is None:
        raise TypeError("Run has no outputs")
    if isinstance(out, dict):
        if set(out.keys()) == {"output"} and isinstance(out["output"], dict):
            return out["output"]
        if isinstance(out.get("output"), dict) and "case_id" in out["output"]:
            return out["output"]
        if "case_id" in out and "workflow" in out:
            return out
    if isinstance(out, str):
        return json.loads(out)
    raise TypeError("Unsupported run output format")


def is_support_case_trace(run):
    trace_name = getattr(run, "name", "") or ""
    if trace_name.startswith("Support case:"):
        return True
    try:
        payload = parse_trace_payload(run)
    except Exception:
        return False
    required_keys = {"workflow", "agent_type", "required_step", "final_answer"}
    return required_keys.issubset(payload.keys())


def matches_eval_filters(run):
    try:
        payload = parse_trace_payload(run)
    except Exception:
        return False
    return (
        payload.get("workflow") in WORKFLOW_FILTERS
        and payload.get("agent_type") in AGENT_TYPE_FILTERS
    )


def run_tags(run):
    tags = getattr(run, "tags", None) or []
    if tags:
        return list(tags)
    extra = getattr(run, "extra", None) or {}
    return list(extra.get("tags") or [])


def run_start_utc(run):
    st = run.start_time
    if st is None:
        return None
    if isinstance(st, str):
        st = datetime.fromisoformat(st.replace("Z", "+00:00"))
    if st.tzinfo is None:
        st = st.replace(tzinfo=timezone.utc)
    return st


def in_time_window(run, from_ts, to_ts):
    st = run_start_utc(run)
    if st is None:
        return True
    return from_ts <= st <= to_ts


def fetch_support_traces(from_timestamp, to_timestamp, verbose=True):
    def query_runs(*, tag_filter: bool, start_time, limit: int):
        kw = dict(
            project_name=PROJECT_NAME,
            start_time=start_time,
            is_root=True,
            limit=limit,
        )
        if tag_filter:
            kw["filter"] = f'has(tags, "{EVAL_TAG}")'
        return list(ls_client.list_runs(**kw))

    candidates = query_runs(tag_filter=True, start_time=from_timestamp, limit=TOTAL_TRACES)
    if verbose and not candidates:
        print(
            "Note: list_runs with has(tags, ...) returned 0 rows; "
            "retrying without the server tag filter and matching tags in Python."
        )
    if not candidates:
        candidates = query_runs(tag_filter=False, start_time=from_timestamp, limit=TOTAL_TRACES)
    if not candidates:
        if verbose:
            print("Note: widening API start_time to the last 7 days to find recent roots.")
        candidates = query_runs(tag_filter=False, start_time=seven_days_ago, limit=TOTAL_TRACES)
    if not candidates:
        if verbose:
            print(
                "Note: listing the most recent root runs with no start_time filter "
                f"(cap {TOTAL_TRACES}); check project name if this is still empty."
            )
        candidates = list(
            ls_client.list_runs(project_name=PROJECT_NAME, is_root=True, limit=TOTAL_TRACES)
        )

    windowed = [r for r in candidates if in_time_window(r, from_timestamp, to_timestamp)]
    if not windowed and candidates:
        tag_only = [r for r in candidates if EVAL_TAG in run_tags(r)]
        if tag_only:
            if verbose:
                print(
                    "Note: no runs fell in the requested time window; "
                    "using tag-matched runs from the query batch instead."
                )
            windowed = tag_only
    tagged = [r for r in windowed if EVAL_TAG in run_tags(r)]
    raw_batch = tagged[:BATCH_SIZE] if tagged else windowed[:BATCH_SIZE]

    support_traces = [t for t in raw_batch if is_support_case_trace(t)]
    filtered_traces = [t for t in support_traces if matches_eval_filters(t)]
    return raw_batch, support_traces, filtered_traces


raw_batch, support_traces, traces_batch = fetch_support_traces(
    from_timestamp=five_am_yesterday,
    to_timestamp=now,
)
if not traces_batch:
    raw_batch, support_traces, traces_batch = fetch_support_traces(
        from_timestamp=seven_days_ago,
        to_timestamp=now,
    )
    print("No matching traces found in the last day, so the search was widened to the last 7 days.")

print(f"Tagged traces fetched: {len(raw_batch)}")
print(f"Support-case traces found: {len(support_traces)}")
print(f"Representative traces in first batch: {len(traces_batch)}")

if not traces_batch:
    raise ValueError(
        "No support-agent traces matched the evaluation filters. "
        "Re-run the logging cells in ch09_example_external_evaluation_pipelines_langsmith.ipynb, then this cell. "
        f"Confirm runs land in project {PROJECT_NAME!r} (same LANGCHAIN_PROJECT as tracing) "
        f"and runs include tag {EVAL_TAG!r}."
    )

traces = [
    {"trace_id": str(run.id), "payload": parse_trace_payload(run)}
    for run in traces_batch
]
print(f"Rows for RULER grouping: {len(traces)}")


Tagged traces fetched: 10
Support-case traces found: 10
Representative traces in first batch: 10
Rows for RULER grouping: 10


In [ ]:
def scenario_group_id(payload):
    return payload.get("scenario_group_id") or (
        f"{payload.get('workflow','unknown')}::{payload.get('required_step','unknown')}::{payload.get('expected_handoff','none')}"
    )

groups = defaultdict(list)
for row in traces:
    groups[scenario_group_id(row["payload"])].append(row)

grouped = {gid: rows[:5] for gid, rows in groups.items() if len(rows) >= 3}
print(f"Comparable groups (>=3 traces): {len(grouped)}")


Comparable groups (>=3 traces): 0


In [ ]:
import asyncio
import nest_asyncio
from art.rewards import ruler


def build_messages(payload):
    return [
        {"role": "system", "content": "Judge answer text quality for customer support."},
        {
            "role": "user",
            "content": (
                f"Customer request: {payload.get('customer_request','')}\n"
                f"Workflow: {payload.get('workflow','')}\n"
                f"Required step: {payload.get('required_step','')}"
            ),
        },
        {"role": "assistant", "content": payload.get("final_answer", "")},
    ]


async def run_ruler(grouped_rows, judge_model="openai/o3"):
    ranked_rows = []
    for gid, rows in grouped_rows.items():
        message_lists = [build_messages(r["payload"]) for r in rows]
        scores = await ruler(
            message_lists,
            judge_model,
            rubric=(
                "Rank by empathy, technical relevance, policy correctness, and actionability. "
                "Prefer clear next steps. Penalize vague answers."
            ),
        )
        tmp = []
        for i, score in enumerate(scores):
            tmp.append(
                {
                    "group_id": gid,
                    "trace_id": rows[i]["trace_id"],
                    "case_id": rows[i]["payload"].get("case_id"),
                    "final_answer": rows[i]["payload"].get("final_answer"),
                    "ruler_score": score.score,
                    "ruler_explanation": score.explanation,
                }
            )
        tmp.sort(key=lambda x: x["ruler_score"], reverse=True)
        for rank, row in enumerate(tmp, start=1):
            row["rank_in_group"] = rank
            row["weak_answer_flag"] = rank == len(tmp)
        ranked_rows.extend(tmp)
    return ranked_rows


nest_asyncio.apply()
ruler_rankings = asyncio.run(run_ruler(grouped))
print(f"Total ranked rows: {len(ruler_rankings)}")


Total ranked rows: 0


In [ ]:
weak_answers = [r for r in ruler_rankings if r["weak_answer_flag"]]
weak_answers[:10]


[]

## Using RULER output in benchmark curation

- Use `weak_answer_flag=True` rows as additional candidates for your benchmark set.
- Keep DeepEval/hard-check failures and RULER weak-answer flags as separate signals.
- Merge by `trace_id`/`case_id` and label source (`deterministic`, `llm_judge`, `comparative_ruler`).
